In [4]:
# ==========================
# 1. Imports
# ==========================
from pathlib import Path
import platform

import duckdb
import pandas as pd

# ==========================
# 1. Import
# ==========================
import numpy as np


# ==========================
# 2. Parâmetros
# ==========================
GESTAO = "LIS"

ANOS = [
    2025,
    2026,
]


# ==========================
# 3. Pasta das bases
# ==========================
def get_db_dir():

    sistema = platform.system()

    if sistema == "Windows":
        return Path(
            r"C:\Users\LISARR\OneDrive - Salvesen Logística S.A\python\00.DB"
        )

    if sistema == "Darwin":
        return Path(
            "/Users/rr/Library/Mobile Documents/"
            "com~apple~CloudDocs/05.Salvesen/00_DB"
        )

    return Path.cwd()


DB_DIR = get_db_dir()


# ==========================
# 4. Caminhos
# ==========================
PATHS = {
    ano: DB_DIR / f"{ano}.duckdb"
    for ano in ANOS
}


# ==========================
# 5. Validar bases
# ==========================
for ano, path in PATHS.items():

    if not path.exists():
        raise FileNotFoundError(
            f"Base {ano} não encontrada: {path}"
        )


# ==========================
# 6. Ligações DuckDB
# ==========================
CON = {
    ano: duckdb.connect(
        str(PATHS[ano]),
        read_only=True
    )
    for ano in ANOS
}


# ==========================
# 7. Validar
# ==========================
validacao = pd.DataFrame({
    "Ano": ANOS,
    "Base": [
        PATHS[ano].name
        for ano in ANOS
    ],
    "Tabelas": [
        len(CON[ano].sql("SHOW TABLES").fetchall())
        for ano in ANOS
    ],
})

validacao


,Ano,Base,Tabelas
0,2025,2025.duckdb,8
1,2026,2026.duckdb,13


In [5]:
# ==================================================
# NORMALIZAÇÃO DOS 18 DISTRITOS DE PORTUGAL
# ==================================================

import pandas as pd
import unicodedata

# 1. Distritos padronizados
distritos = [
    "Aveiro",
    "Beja",
    "Braga",
    "Bragança",
    "Castelo Branco",
    "Coimbra",
    "Évora",
    "Faro",
    "Guarda",
    "Leiria",
    "Lisboa",
    "Portalegre",
    "Porto",
    "Santarém",
    "Setúbal",
    "Viana do Castelo",
    "Vila Real",
    "Viseu"
]

# 2. Correspondência entre localidades e distritos
grupos = {
    "Aveiro": [
        "Aveiro",
        "Albergaria-a-Velha",
        "Anadia",
        "Arouca",
        "Espinho",
        "Estarreja",
        "Mealhada",
        "Murtosa",
        "Oliveira de Azeméis",
        "Oliveira do Bairro",
        "Ovar",
        "Santa Maria da Feira",
        "Sever do Vouga",
        "São João da Madeira",
        "Vale de Cambra",
        "Águeda",
        "Ílhavo"
    ],

    "Beja": ["Beja", "Castro Verde"],
    "Braga": ["Braga"],
    "Bragança": ["Bragança"],
    "Castelo Branco": ["Castelo Branco"],
    "Coimbra": ["Coimbra"],
    "Évora": ["Évora"],
    "Faro": ["Faro"],
    "Guarda": ["Guarda"],
    "Leiria": ["Leiria"],
    "Lisboa": ["Lisboa"],
    "Portalegre": ["Portalegre"],
    "Porto": ["Porto"],
    "Santarém": ["Santarém", "Ourém"],
    "Setúbal": ["Setúbal"],
    "Viana do Castelo": ["Viana do Castelo"],
    "Vila Real": ["Vila Real"],
    "Viseu": ["Viseu"]
}

# 3. Função de normalização
def normalizar(texto):

    if pd.isna(texto):
        return ""

    texto = unicodedata.normalize(
        "NFKD",
        str(texto).strip()
    )

    return "".join(
        c for c in texto
        if not unicodedata.combining(c)
    ).upper()

# 4. Criar mapa de correspondência
mapa = {
    normalizar(localidade): distrito
    for distrito, localidades in grupos.items()
    for localidade in localidades
}

# 5. Validar
print("Normalização configurada com sucesso.")
print(f"Distritos: {len(distritos)}")
print(f"Localidades mapeadas: {len(mapa)}")

Normalização configurada com sucesso.
Distritos: 18
Localidades mapeadas: 36


In [9]:
# ==========================================================
# PERFIL OPERACIONAL E ECONÓMICO DOS CLIENTES
# JULHO, AGOSTO E SETEMBRO DE 2026
# DESTINOS IDENTIFICADOS PELO CAMPO LOCDES
# ==========================================================

import pandas as pd
import numpy as np

# ==========================
# 1. CONFIGURAÇÃO
# ==========================

ANO = 2026
MESES = ["202607", "202608", "202609"]

# None = todos os clientes
# CLIENTES = ["391", "404", "405", "407", "415", "551", "680", "757", "311"]

CLIENTES = None

# ==========================
# 2. EXTRAIR DADOS
# ==========================

filtro_clientes = ""
parametros = [GESTAO, *MESES]

if CLIENTES is not None:
    filtro_clientes = f"""
        AND LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0')
        IN ({','.join(['?'] * len(CLIENTES))})
    """
    parametros.extend(CLIENTES)

dados = CON[ANO].execute(
    f"""
    SELECT
        FENTREGA,
        LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') AS CODACT,
        ACTIVIDAD,
        REFERENCIA,
        LOCDES,
        PROV_DESTINO,
        PALETS,
        PESO_BRUTO,
        INGRESODT,
        COSTEDT

    FROM inform_27_{ANO}

    WHERE Gestion = ?

      AND SUBSTR(
          TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6
      ) IN (?, ?, ?)

      {filtro_clientes}
    """,
    parametros
).df()

# ==========================
# 3. PREPARAR DADOS
# ==========================

dados = dados.reset_index(drop=True)

dados["MES"] = (
    dados["FENTREGA"]
    .astype("string")
    .str.strip()
    .str[:6]
)

for coluna in [
    "PALETS",
    "PESO_BRUTO",
    "INGRESODT",
    "COSTEDT"
]:
    dados[coluna] = pd.to_numeric(
        dados[coluna]
        .astype("string")
        .str.replace(",", ".", regex=False),
        errors="coerce"
    )

# Normalizar LOCDES e REFERENCIA
for coluna in ["LOCDES", "REFERENCIA"]:

    dados[coluna] = (
        dados[coluna]
        .astype("string")
        .str.strip()
    )

    dados[coluna] = dados[coluna].mask(
        dados[coluna].eq(""),
        pd.NA
    )

# Distrito padronizado
dados["DISTRITO"] = (
    dados["PROV_DESTINO"]
    .map(normalizar)
    .map(mapa)
    .fillna("NÃO MAPEADO")
)

# ==========================
# 4. IDENTIFICAR ENTREGAS
# ==========================

dados["ID_ENTREGA"] = dados["REFERENCIA"].copy()

# Sem referência: manter cada linha como registo separado
sem_referencia = dados["ID_ENTREGA"].isna()

dados.loc[sem_referencia, "ID_ENTREGA"] = (
    "SEM_REF_LINHA_"
    + dados.index[sem_referencia].astype(str)
)

# Agrupar linhas pertencentes à mesma entrega
df_entregas = (
    dados.groupby(
        [
            "MES",
            "CODACT",
            "FENTREGA",
            "ID_ENTREGA",
            "LOCDES",
            "DISTRITO"
        ],
        dropna=False,
        as_index=False
    )[
        [
            "PALETS",
            "PESO_BRUTO",
            "INGRESODT",
            "COSTEDT"
        ]
    ]
    .sum()
)

# ==========================
# 5. CRIAR PERÍODOS
# ==========================

# Dados mensais
mensal = df_entregas.rename(
    columns={"MES": "PERIODO"}
).copy()

# Dados consolidados dos três meses
total = mensal.copy()
total["PERIODO"] = "TOTAL JUL-SET"

# Base para calcular ambos os perfis
base = pd.concat(
    [mensal, total],
    ignore_index=True
)

# ==========================
# 6. PERFIL POR LOCDES
# ==========================

# Excluir LOCDES vazios apenas da contagem de destinos
base_destinos = base[
    base["LOCDES"].notna()
].copy()

df_destinos = (
    base_destinos.groupby(
        ["PERIODO", "CODACT", "LOCDES"],
        as_index=False
    )
    .agg(
        N_ENTREGAS=("ID_ENTREGA", "size"),
        PALETS=("PALETS", "sum"),
        PESO_BRUTO=("PESO_BRUTO", "sum"),
        INGRESODT=("INGRESODT", "sum"),
        COSTEDT=("COSTEDT", "sum")
    )
)

# Média de paletes por entrega em cada LOCDES
df_destinos["PALETES_MEDIA_ENTREGA"] = (
    df_destinos["PALETS"]
    / df_destinos["N_ENTREGAS"].replace(0, np.nan)
)

# Identificar destinos com média de 1 a 2 paletes
df_destinos["DESTINO_1_A_2_PALETES"] = (
    df_destinos["PALETES_MEDIA_ENTREGA"]
    .between(1, 2, inclusive="both")
)

# Resumo dos destinos por cliente
resumo_destinos = (
    df_destinos.groupby(
        ["PERIODO", "CODACT"],
        as_index=False
    )
    .agg(
        DESTINOS_DIFERENTES=("LOCDES", "nunique"),
        DESTINOS_1_A_2=("DESTINO_1_A_2_PALETES", "sum"),
        PALETES_DESTINOS=("PALETS", "sum")
    )
)

# ==========================
# 7. PERFIL GERAL DO CLIENTE
# ==========================

df_perfil = (
    base.groupby(
        ["PERIODO", "CODACT"],
        as_index=False
    )
    .agg(
        N_ENTREGAS=("ID_ENTREGA", "size"),
        PALETS=("PALETS", "sum"),
        PESO_BRUTO=("PESO_BRUTO", "sum"),
        INGRESODT=("INGRESODT", "sum"),
        COSTEDT=("COSTEDT", "sum")
    )
)

# Acrescentar indicadores dos destinos
df_perfil = df_perfil.merge(
    resumo_destinos,
    on=["PERIODO", "CODACT"],
    how="left"
)

for coluna in [
    "DESTINOS_DIFERENTES",
    "DESTINOS_1_A_2",
    "PALETES_DESTINOS"
]:
    df_perfil[coluna] = df_perfil[coluna].fillna(0)

# ==========================
# 8. CALCULAR INDICADORES
# ==========================

# Paletes médias por destino LOCDES
df_perfil["PALETES_MEDIA_DESTINO"] = (
    df_perfil["PALETES_DESTINOS"]
    / df_perfil["DESTINOS_DIFERENTES"].replace(0, np.nan)
)

# Peso por palete
df_perfil["PESO_POR_PALETE"] = (
    df_perfil["PESO_BRUTO"]
    / df_perfil["PALETS"].replace(0, np.nan)
)

# Ingresso por palete
df_perfil["INGRESSO_POR_PALETE"] = (
    df_perfil["INGRESODT"]
    / df_perfil["PALETS"].replace(0, np.nan)
)

# Custo por palete
df_perfil["CUSTO_POR_PALETE"] = (
    df_perfil["COSTEDT"]
    / df_perfil["PALETS"].replace(0, np.nan)
)

# Percentagem de destinos LOCDES com média de 1 a 2 paletes
df_perfil["PCT_DESTINOS_1_A_2"] = (
    df_perfil["DESTINOS_1_A_2"]
    / df_perfil["DESTINOS_DIFERENTES"].replace(0, np.nan)
    * 100
)

# ==========================
# 9. ACRESCENTAR NOME DO CLIENTE
# ==========================

nomes_clientes = (
    dados.groupby("CODACT")["ACTIVIDAD"]
    .agg(
        lambda s: (
            s.mode().iloc[0]
            if not s.mode().empty
            else ""
        )
    )
    .reset_index()
)

df_perfil = df_perfil.merge(
    nomes_clientes,
    on="CODACT",
    how="left"
)

# Organizar colunas
colunas_perfil = [
    "PERIODO",
    "CODACT",
    "ACTIVIDAD",
    "DESTINOS_DIFERENTES",
    "N_ENTREGAS",
    "PALETS",
    "PALETES_MEDIA_DESTINO",
    "PESO_BRUTO",
    "PESO_POR_PALETE",
    "INGRESODT",
    "COSTEDT",
    "INGRESSO_POR_PALETE",
    "CUSTO_POR_PALETE",
    "PCT_DESTINOS_1_A_2"
]

df_perfil = df_perfil[colunas_perfil]

# Arredondar indicadores
colunas_arredondar = [
    "PALETS",
    "PALETES_MEDIA_DESTINO",
    "PESO_BRUTO",
    "PESO_POR_PALETE",
    "INGRESODT",
    "COSTEDT",
    "INGRESSO_POR_PALETE",
    "CUSTO_POR_PALETE",
    "PCT_DESTINOS_1_A_2"
]

df_perfil[colunas_arredondar] = (
    df_perfil[colunas_arredondar].round(2)
)

# ==========================
# 10. ENTREGAS POR DISTRITO
# ==========================

df_entregas_distrito = (
    base.groupby(
        ["PERIODO", "CODACT", "DISTRITO"],
        as_index=False
    )
    .agg(
        N_ENTREGAS=("ID_ENTREGA", "size")
    )
)

# Criar matriz com um distrito por coluna
df_matriz_distritos = (
    df_entregas_distrito.pivot_table(
        index=["PERIODO", "CODACT"],
        columns="DISTRITO",
        values="N_ENTREGAS",
        aggfunc="sum",
        fill_value=0
    )
    .reindex(
        columns=distritos + ["NÃO MAPEADO"],
        fill_value=0
    )
)

df_matriz_distritos["TOTAL_ENTREGAS"] = (
    df_matriz_distritos.sum(axis=1)
)

df_matriz_distritos = (
    df_matriz_distritos.reset_index()
)

df_matriz_distritos.columns.name = None

# Acrescentar nome do cliente
df_matriz_distritos = df_matriz_distritos.merge(
    nomes_clientes,
    on="CODACT",
    how="left"
)

# ==========================
# 11. SEPARAR RESULTADOS
# ==========================

# Perfil consolidado: julho + agosto + setembro
df_perfil_total = (
    df_perfil[
        df_perfil["PERIODO"] == "TOTAL JUL-SET"
    ]
    .drop(columns="PERIODO")
    .sort_values("CODACT")
    .reset_index(drop=True)
)

# Perfil mensal
df_perfil_mensal = (
    df_perfil[
        df_perfil["PERIODO"] != "TOTAL JUL-SET"
    ]
    .sort_values(["CODACT", "PERIODO"])
    .reset_index(drop=True)
)

# Entregas por distrito - total dos 3 meses
df_distritos_total = (
    df_matriz_distritos[
        df_matriz_distritos["PERIODO"] == "TOTAL JUL-SET"
    ]
    .drop(columns="PERIODO")
    .sort_values("CODACT")
    .reset_index(drop=True)
)

# Entregas por distrito - mês a mês
df_distritos_mensal = (
    df_matriz_distritos[
        df_matriz_distritos["PERIODO"] != "TOTAL JUL-SET"
    ]
    .sort_values(["CODACT", "PERIODO"])
    .reset_index(drop=True)
)

# ==========================
# 12. VISUALIZAR RESULTADOS
# ==========================

print("=" * 60)
print("PERFIL CONSOLIDADO - JULHO A SETEMBRO 2026")
print("IDENTIFICAÇÃO DOS DESTINOS: LOCDES")
print("=" * 60)

display(df_perfil_total)

print("\nPERFIL MENSAL")
display(df_perfil_mensal)

print("\nENTREGAS POR DISTRITO - TOTAL DOS 3 MESES")
display(df_distritos_total)

print("\nENTREGAS POR DISTRITO - MÊS A MÊS")
display(df_distritos_mensal)

print("\nCONTROLO DOS DADOS")
print(f"Linhas originais: {len(dados):,}")
print(f"Entregas identificadas: {len(df_entregas):,}")
print(f"Linhas sem referência: {sem_referencia.sum():,}")
print(f"Clientes analisados: {dados['CODACT'].nunique():,}")

PERFIL CONSOLIDADO - JULHO A SETEMBRO 2026
IDENTIFICAÇÃO DOS DESTINOS: LOCDES


,CODACT,ACTIVIDAD,DESTINOS_DIFERENTES,N_ENTREGAS,PALETS,PALETES_MEDIA_DESTINO,PESO_BRUTO,PESO_POR_PALETE,INGRESODT,COSTEDT,INGRESSO_POR_PALETE,CUSTO_POR_PALETE,PCT_DESTINOS_1_A_2
0,011,DANONE PORTUGAL,236,6720,50623.0,214.48,15161731.89,299.5,198082.08,468897.31,3.91,9.26,77.97
1,013,DANONE PORTUGAL CAPILAR,627,9576,9618.0,15.34,793184.83,82.47,0.0,183523.3,0.0,19.08,99.2
2,015,TROPICANA,32,478,819.0,25.59,311218.54,380.0,0.0,13066.47,0.0,15.95,46.88
3,028,PLN,3,31,71.0,23.67,34494.2,485.83,171.14,1523.98,2.41,21.46,66.67
4,074,NUTRICIA,771,8113,12802.0,16.58,1802230.48,140.78,155227.08,166451.31,12.13,13.0,89.62
5,084,DANONE PORTUGAL TLD,2,60,1962.0,981.0,0.0,0.0,8092.29,8018.0,4.12,4.09,0.0
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,4,5,19.0,4.75,12218.04,643.05,0.0,348.0,0.0,18.32,50.0
7,117,ORANGINA SCHWEPPES,10,48,576.0,57.6,378423.0,656.98,0.0,14824.31,0.0,25.74,0.0
8,118,ORANGNA SCHWEPPES PORTUGAL,26,644,1704.0,65.54,960917.37,563.92,85983.99,25966.91,50.46,15.24,26.92
9,131,RANA,115,1559,3782.0,32.89,545428.42,144.22,59818.44,30189.03,15.82,7.98,82.61



PERFIL MENSAL


,PERIODO,CODACT,ACTIVIDAD,DESTINOS_DIFERENTES,N_ENTREGAS,PALETS,PALETES_MEDIA_DESTINO,PESO_BRUTO,PESO_POR_PALETE,INGRESODT,COSTEDT,INGRESSO_POR_PALETE,CUSTO_POR_PALETE,PCT_DESTINOS_1_A_2
0,202607,011,DANONE PORTUGAL,198,2421,16358.0,82.61,5452883.44,333.35,36230.55,172825.65,2.21,10.57,81.82
1,202608,011,DANONE PORTUGAL,174,2101,18342.0,105.39,4928839.05,268.72,81139.92,159709.89,4.42,8.71,79.31
2,202609,011,DANONE PORTUGAL,166,2198,15923.0,95.92,4780009.4,300.2,80711.61,136361.77,5.07,8.56,66.87
3,202607,013,DANONE PORTUGAL CAPILAR,613,3343,3365.0,5.49,291310.33,86.57,0.0,64671.24,0.0,19.22,100.0
4,202608,013,DANONE PORTUGAL CAPILAR,581,3037,3046.0,5.24,245060.87,80.45,0.0,56853.64,0.0,18.67,100.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,202607,860,ANDROS PTG CONGELADO,2,4,6.0,3.0,1374.66,229.11,135.8,51.07,22.63,8.51,100.0
134,202609,860,ANDROS PTG CONGELADO,1,1,1.0,1.0,37.21,37.21,18.75,1.32,18.75,1.32,100.0
135,202607,AAB,LAS LAGUNAS DE SANCHONUÑO (HUERCASA),6,52,261.0,43.5,130000.0,498.08,0.0,4207.74,0.0,16.12,16.67
136,202608,AAB,LAS LAGUNAS DE SANCHONUÑO (HUERCASA),6,48,222.0,37.0,117500.0,529.28,0.0,4114.47,0.0,18.53,16.67



ENTREGAS POR DISTRITO - TOTAL DOS 3 MESES


,CODACT,Aveiro,Beja,Braga,Bragança,Castelo Branco,Coimbra,Évora,Faro,Guarda,...,Portalegre,Porto,Santarém,Setúbal,Viana do Castelo,Vila Real,Viseu,NÃO MAPEADO,TOTAL_ENTREGAS,ACTIVIDAD
0,011,427,37,314,76,47,274,13,259,0,...,27,2470,306,290,107,5,0,52,6720,DANONE PORTUGAL
1,013,846,9,815,0,0,137,0,12,14,...,0,835,106,185,14,0,38,6070,9576,DANONE PORTUGAL CAPILAR
2,015,28,0,0,0,0,34,0,38,0,...,0,152,22,6,0,0,0,20,478,TROPICANA
3,028,0,0,0,0,0,0,0,0,0,...,0,28,0,0,0,0,0,0,31,PLN
4,074,422,135,544,67,87,566,77,528,72,...,35,2543,274,367,92,63,109,157,8113,NUTRICIA
5,084,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,60,DANONE PORTUGAL TLD
6,116,0,0,0,0,0,0,0,0,0,...,0,3,0,0,0,0,0,0,5,ORANGINA SCHWEPPES ESPAÑA BASE
7,117,10,0,0,0,0,0,0,0,15,...,0,2,3,4,0,0,0,6,48,ORANGINA SCHWEPPES
8,118,73,0,0,0,0,0,0,30,0,...,0,192,76,45,0,0,0,6,644,ORANGNA SCHWEPPES PORTUGAL
9,131,7,0,8,0,2,62,0,141,0,...,0,511,102,66,0,1,1,20,1559,RANA



ENTREGAS POR DISTRITO - MÊS A MÊS


,PERIODO,CODACT,Aveiro,Beja,Braga,Bragança,Castelo Branco,Coimbra,Évora,Faro,...,Portalegre,Porto,Santarém,Setúbal,Viana do Castelo,Vila Real,Viseu,NÃO MAPEADO,TOTAL_ENTREGAS,ACTIVIDAD
0,202607,011,149,15,113,26,18,85,6,92,...,9,889,114,104,39,3,0,15,2421,DANONE PORTUGAL
1,202608,011,140,13,101,23,19,100,5,82,...,9,781,94,100,32,1,0,18,2101,DANONE PORTUGAL
2,202609,011,138,9,100,27,10,89,2,85,...,9,800,98,86,36,1,0,19,2198,DANONE PORTUGAL
3,202607,013,298,5,283,0,0,49,0,10,...,0,293,39,69,5,0,14,2089,3343,DANONE PORTUGAL CAPILAR
4,202608,013,271,2,259,0,0,40,0,1,...,0,270,34,62,4,0,12,1918,3037,DANONE PORTUGAL CAPILAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,202607,860,0,0,0,0,0,0,0,0,...,0,1,3,0,0,0,0,0,4,ANDROS PTG CONGELADO
134,202609,860,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,1,ANDROS PTG CONGELADO
135,202607,AAB,0,0,0,0,0,0,0,9,...,0,19,0,5,0,0,0,0,52,LAS LAGUNAS DE SANCHONUÑO (HUERCASA)
136,202608,AAB,0,0,0,0,0,0,0,9,...,0,17,0,4,0,0,0,0,48,LAS LAGUNAS DE SANCHONUÑO (HUERCASA)



CONTROLO DOS DADOS
Linhas originais: 50,078
Entregas identificadas: 49,403
Linhas sem referência: 0
Clientes analisados: 49


In [10]:
# ==================================================
# EXPORTAR PERFIL DOS CLIENTES PARA EXCEL
# JULHO, AGOSTO E SETEMBRO DE 2026
# ==================================================

from pathlib import Path
import pandas as pd

# Localização do ficheiro
ficheiro = (
    Path.home()
    / "Documents"
    / "Perfil_Clientes_Julho_Setembro_2026.xlsx"
)

# Exportar todas as análises para um único Excel
with pd.ExcelWriter(ficheiro, engine="openpyxl") as writer:

    df_perfil_total.to_excel(
        writer,
        sheet_name="Perfil Consolidado",
        index=False
    )

    df_perfil_mensal.to_excel(
        writer,
        sheet_name="Perfil Mensal",
        index=False
    )

    df_distritos_total.to_excel(
        writer,
        sheet_name="Entregas Distrito Total",
        index=False
    )

    df_distritos_mensal.to_excel(
        writer,
        sheet_name="Entregas Distrito Mensal",
        index=False
    )

    df_destinos.to_excel(
        writer,
        sheet_name="Analise Destinos",
        index=False
    )

    df_entregas.to_excel(
        writer,
        sheet_name="Detalhe Entregas",
        index=False
    )

    # Ajustar largura das colunas e aplicar filtros
    for folha in writer.sheets.values():

        folha.freeze_panes = "A2"
        folha.auto_filter.ref = folha.dimensions

        for coluna in folha.columns:
            comprimento = max(
                len(str(celula.value))
                if celula.value is not None else 0
                for celula in coluna
            )

            folha.column_dimensions[
                coluna[0].column_letter
            ].width = min(comprimento + 3, 40)

print(f"Excel exportado com sucesso:\n{ficheiro}")

Excel exportado com sucesso:
C:\Users\LISARR\Documents\Perfil_Clientes_Julho_Setembro_2026.xlsx


In [8]:
# VALIDAR SE DESTINO E LOCDES SÃO EQUIVALENTES
# JULHO, AGOSTO E SETEMBRO DE 2026

df_validacao = CON[2026].execute("""
    SELECT
        LPAD(TRIM(CAST(CODACT AS VARCHAR)), 3, '0') AS CODACT,
        ACTIVIDAD,
        DESTINO,
        LOCDES
    FROM inform_27_2026
    WHERE Gestion = ?
      AND SUBSTR(TRIM(CAST(FENTREGA AS VARCHAR)), 1, 6)
          IN ('202607', '202608', '202609')
""", [GESTAO]).df()

# Normalizar valores
for col in ["DESTINO", "LOCDES"]:
    df_validacao[col] = (
        df_validacao[col]
        .astype("string")
        .str.strip()
        .str.upper()
        .replace("", pd.NA)
    )

# Comparar destinos distintos por cliente
resultado = (
    df_validacao.groupby(["CODACT", "ACTIVIDAD"], dropna=False)
    .agg(
        DESTINOS_DESTINO=("DESTINO", "nunique"),
        DESTINOS_LOCDES=("LOCDES", "nunique")
    )
    .reset_index()
)

# Verificar se existe correspondência 1 para 1
mapa_destino = (
    df_validacao.groupby(["CODACT", "DESTINO"], dropna=True)["LOCDES"]
    .nunique()
    .groupby(level=0)
    .max()
)

mapa_locdes = (
    df_validacao.groupby(["CODACT", "LOCDES"], dropna=True)["DESTINO"]
    .nunique()
    .groupby(level=0)
    .max()
)

resultado["DIFERENCA"] = (
    resultado["DESTINOS_DESTINO"] - resultado["DESTINOS_LOCDES"]
)

resultado["EQUIVALENTES"] = (
    resultado["DIFERENCA"].eq(0)
    & resultado["CODACT"].map(mapa_destino).fillna(0).le(1)
    & resultado["CODACT"].map(mapa_locdes).fillna(0).le(1)
    & ~resultado["CODACT"].isin(
        df_validacao.loc[
            df_validacao["DESTINO"].isna() != df_validacao["LOCDES"].isna(),
            "CODACT"
        ]
    )
)

display(resultado)

print(
    "RESULTADO:",
    "EQUIVALENTES" if resultado["EQUIVALENTES"].all()
    else "EXISTEM DIFERENÇAS ENTRE DESTINO E LOCDES"
)

,CODACT,ACTIVIDAD,DESTINOS_DESTINO,DESTINOS_LOCDES,DIFERENCA,EQUIVALENTES
0,011,DANONE PORTUGAL,256,236,20,False
1,013,DANONE PORTUGAL CAPILAR,622,627,-5,False
2,015,TROPICANA,30,32,-2,False
3,028,PLN,2,3,-1,False
4,074,NUTRICIA,762,771,-9,False
5,084,DANONE PORTUGAL TLD,2,2,0,True
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,4,4,0,True
7,117,ORANGINA SCHWEPPES,10,10,0,True
8,118,ORANGNA SCHWEPPES PORTUGAL,30,26,4,False
9,131,RANA,114,115,-1,False


RESULTADO: EXISTEM DIFERENÇAS ENTRE DESTINO E LOCDES


In [11]:
# ==========================================================
# PERFIL CONSOLIDADO - AGOSTO + SETEMBRO 2026
# EXPORTAR PARA EXCEL (UMA FOLHA)
# ==========================================================

from pathlib import Path
import pandas as pd
import numpy as np

# 1. Filtrar agosto e setembro
df_periodo = df_entregas[
    df_entregas["MES"].isin(["202608", "202609"])
].copy()

# 2. Analisar destinos LOCDES
df_locdes = (
    df_periodo[df_periodo["LOCDES"].notna()]
    .groupby(["CODACT", "LOCDES"], as_index=False)
    .agg(
        N_ENTREGAS=("ID_ENTREGA", "size"),
        PALETS=("PALETS", "sum")
    )
)

df_locdes["MEDIA_PALETES_ENTREGA"] = (
    df_locdes["PALETS"] / df_locdes["N_ENTREGAS"]
)

df_locdes["DESTINO_1_A_2"] = (
    df_locdes["MEDIA_PALETES_ENTREGA"].between(1, 2)
)

resumo_destinos = (
    df_locdes.groupby("CODACT", as_index=False)
    .agg(
        DESTINOS_DIFERENTES=("LOCDES", "nunique"),
        DESTINOS_1_A_2=("DESTINO_1_A_2", "sum"),
        PALETES_DESTINOS=("PALETS", "sum")
    )
)

# 3. Consolidar os dois meses por cliente
df_consolidado = (
    df_periodo.groupby("CODACT", as_index=False)
    .agg(
        N_ENTREGAS=("ID_ENTREGA", "size"),
        PALETS=("PALETS", "sum"),
        PESO_BRUTO=("PESO_BRUTO", "sum"),
        INGRESODT=("INGRESODT", "sum"),
        COSTEDT=("COSTEDT", "sum")
    )
)

df_consolidado = (
    df_consolidado
    .merge(nomes_clientes, on="CODACT", how="left")
    .merge(resumo_destinos, on="CODACT", how="left")
)

# 4. Calcular indicadores
df_consolidado["PALETES_MEDIA_DESTINO"] = (
    df_consolidado["PALETES_DESTINOS"]
    / df_consolidado["DESTINOS_DIFERENTES"].replace(0, np.nan)
)

df_consolidado["PESO_POR_PALETE"] = (
    df_consolidado["PESO_BRUTO"]
    / df_consolidado["PALETS"].replace(0, np.nan)
)

df_consolidado["INGRESSO_POR_PALETE"] = (
    df_consolidado["INGRESODT"]
    / df_consolidado["PALETS"].replace(0, np.nan)
)

df_consolidado["CUSTO_POR_PALETE"] = (
    df_consolidado["COSTEDT"]
    / df_consolidado["PALETS"].replace(0, np.nan)
)

df_consolidado["PCT_DESTINOS_1_A_2"] = (
    df_consolidado["DESTINOS_1_A_2"]
    / df_consolidado["DESTINOS_DIFERENTES"].replace(0, np.nan)
    * 100
)

df_consolidado["MARGEM_EUR"] = (
    df_consolidado["INGRESODT"] - df_consolidado["COSTEDT"]
)

df_consolidado["MARGEM_PCT"] = (
    df_consolidado["MARGEM_EUR"]
    / df_consolidado["INGRESODT"].replace(0, np.nan)
    * 100
)

# 5. Organizar colunas
colunas = [
    "CODACT",
    "ACTIVIDAD",
    "DESTINOS_DIFERENTES",
    "N_ENTREGAS",
    "PALETS",
    "PALETES_MEDIA_DESTINO",
    "PESO_BRUTO",
    "PESO_POR_PALETE",
    "INGRESODT",
    "COSTEDT",
    "INGRESSO_POR_PALETE",
    "CUSTO_POR_PALETE",
    "PCT_DESTINOS_1_A_2",
    "MARGEM_EUR",
    "MARGEM_PCT"
]

df_consolidado = (
    df_consolidado[colunas]
    .sort_values("CODACT")
    .reset_index(drop=True)
)

df_consolidado = df_consolidado.round(2)

# 6. Exportar para Excel
ficheiro = (
    Path.home()
    / "Documents"
    / "Perfil_Consolidado_Agosto_Setembro_2026.xlsx"
)

ficheiro.parent.mkdir(parents=True, exist_ok=True)

with pd.ExcelWriter(ficheiro, engine="openpyxl") as writer:

    df_consolidado.to_excel(
        writer,
        sheet_name="Perfil Consolidado",
        index=False
    )

    folha = writer.sheets["Perfil Consolidado"]
    folha.freeze_panes = "C2"
    folha.auto_filter.ref = folha.dimensions

    for coluna in folha.columns:
        largura = max(
            len(str(celula.value)) if celula.value is not None else 0
            for celula in coluna
        )

        folha.column_dimensions[
            coluna[0].column_letter
        ].width = min(largura + 3, 42)

print(f"Excel criado com sucesso:\n{ficheiro}")

display(df_consolidado)

Excel criado com sucesso:
C:\Users\LISARR\Documents\Perfil_Consolidado_Agosto_Setembro_2026.xlsx


,CODACT,ACTIVIDAD,DESTINOS_DIFERENTES,N_ENTREGAS,PALETS,PALETES_MEDIA_DESTINO,PESO_BRUTO,PESO_POR_PALETE,INGRESODT,COSTEDT,INGRESSO_POR_PALETE,CUSTO_POR_PALETE,PCT_DESTINOS_1_A_2,MARGEM_EUR,MARGEM_PCT
0,011,DANONE PORTUGAL,215,4299,34265.0,159.35,9708848.45,283.35,161851.53,296071.66,4.72,8.64,74.88,-134220.13,-82.93
1,013,DANONE PORTUGAL CAPILAR,607,6233,6253.0,10.3,501874.5,80.26,0.0,118852.06,0.0,19.01,99.18,-118852.06,<NA>
2,015,TROPICANA,30,295,436.0,14.53,164997.98,378.44,0.0,6912.71,0.0,15.85,60.0,-6912.71,<NA>
3,028,PLN,3,25,53.0,17.67,23957.22,452.02,171.14,1113.05,3.23,21.0,66.67,-941.91,-550.37
4,074,NUTRICIA,731,5461,8633.0,11.78,1214173.9,140.64,104530.51,112525.0,12.11,13.03,88.65,-7994.49,-7.65
5,084,DANONE PORTUGAL TLD,1,35,1155.0,1155.0,0.0,0.0,4968.95,4950.0,4.3,4.29,0.0,18.95,0.38
6,116,ORANGINA SCHWEPPES ESPAÑA BASE,2,2,14.0,7.0,10250.77,732.2,0.0,120.64,0.0,8.62,0.0,-120.64,<NA>
7,117,ORANGINA SCHWEPPES,9,24,288.0,32.0,191189.69,663.85,0.0,6954.77,0.0,24.15,0.0,-6954.77,<NA>
8,118,ORANGNA SCHWEPPES PORTUGAL,26,429,1110.0,42.69,626109.17,564.06,57951.47,16543.41,52.21,14.9,42.31,41408.06,71.45
9,131,RANA,101,1034,2522.0,24.97,400812.31,158.93,39749.92,20473.85,15.76,8.12,80.2,19276.07,48.49


In [13]:
# ==========================================================
# ENTREGAS TOTAIS VS ENTREGAS DE 1 PALETE COM PESO < 100 KG
# AGOSTO E SETEMBRO DE 2026
# ==========================================================

# Base: entregas de agosto e setembro
df_analise = df_entregas[
    df_entregas["MES"].isin(["202608", "202609"])
].copy()

# Identificar entregas de 1 palete com peso entre 0 e 100 kg
df_analise["ENTREGA_1_PALETE_MENOS_100KG"] = (
    (df_analise["PALETS"] == 1) &
    (df_analise["PESO_BRUTO"] > 0) &
    (df_analise["PESO_BRUTO"] < 100)
)

# Agrupar por cliente
df_resultado = (
    df_analise.groupby("CODACT", as_index=False)
    .agg(
        TOTAL_ENTREGAS=("ID_ENTREGA", "size"),
        ENTREGAS_1_PALETE_MENOS_100KG=(
            "ENTREGA_1_PALETE_MENOS_100KG", "sum"
        )
    )
)

# Calcular percentagem
df_resultado["PERCENTAGEM"] = (
    df_resultado["ENTREGAS_1_PALETE_MENOS_100KG"]
    / df_resultado["TOTAL_ENTREGAS"]
    * 100
).round(2)

# Acrescentar nome do cliente
df_resultado = df_resultado.merge(
    nomes_clientes,
    on="CODACT",
    how="left"
)

# Organizar colunas
df_resultado = df_resultado[
    [
        "CODACT",
        "ACTIVIDAD",
        "TOTAL_ENTREGAS",
        "ENTREGAS_1_PALETE_MENOS_100KG",
        "PERCENTAGEM"
    ]
].sort_values(
    "ENTREGAS_1_PALETE_MENOS_100KG",
    ascending=False
).reset_index(drop=True)

display(df_resultado)

,CODACT,ACTIVIDAD,TOTAL_ENTREGAS,ENTREGAS_1_PALETE_MENOS_100KG,PERCENTAGEM
0,013,DANONE PORTUGAL CAPILAR,6233,4632,74.31
1,074,NUTRICIA,5461,4033,73.85
2,311,ALPRO PORTUGAL,4792,3530,73.66
3,011,DANONE PORTUGAL,4299,1137,26.45
4,131,RANA,1034,641,61.99
5,332,JERVIS,598,475,79.43
6,260,ANDROS REFRIGERADO,907,323,35.61
7,338,ARIAS STOCK,541,298,55.08
8,404,RTM,573,289,50.44
9,407,NUVIFRUITS,549,241,43.9
